# 06.9 - Checkpointing & Transfer Learning

**Phase:** 06 - Deep Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

Checkpointing saves model params, optimizer state, and training progress so training can resume or the best model can be recovered. Transfer learning reuses a pretrained model on a new task, cutting data and compute needs.

## 2. Why Does This Matter?

Training large models takes hours to days; without checkpoints a crash means starting over. Without transfer learning, real-world tasks need impractical data and compute.

## 3. Prerequisites

- Unit 06.8 (training loops), basic file I/O

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Save and load model checkpoints correctly
- Implement transfer learning (feature extraction + fine-tuning)
- Resume training from a saved checkpoint

## 5. Mental Model

Checkpointing is saving your game progress. Transfer learning is hiring someone who already knows most of the job and just needs the specifics.

> NO torchvision pretrained models used here: we build a small custom CNN as the 'pretrained' source and fine-tune on synthetic data, per the phase constraints.


## 6. Backend


In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import os

torch.manual_seed(42); np.random.seed(42)
print("Ready.")


Ready.


## 7. Baseline Checkpointing

Save and load a small model's `state_dict`, verifying predictions match.


In [2]:
model = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 2))
x = torch.randn(5, 8)
with torch.no_grad():
    before = model(x)

save_path = os.path.join(os.getcwd(), '_tmp_ckpt.pt')
torch.save(model.state_dict(), save_path)

model2 = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 2))
model2.load_state_dict(torch.load(save_path, map_location='cpu'))
with torch.no_grad():
    after = model2(x)

print("Predictions match after save/load:", torch.allclose(before, after))
print("Saved file exists:", os.path.exists(save_path))


Predictions match after save/load: True
Saved file exists: True


## 8. Save Optimizer State to Resume Training

A full checkpoint bundles epoch, model, optimizer, and loss.


In [3]:
opt = optim.Adam(model.parameters(), lr=0.01)
os.makedirs('_tmp_runs', exist_ok=True)
ckpt = {'epoch': 5, 'model_state_dict': model.state_dict(),
        'optimizer_state_dict': opt.state_dict(), 'loss': 0.123}
torch.save(ckpt, '_tmp_runs/full.pt')

loaded = torch.load('_tmp_runs/full.pt', map_location='cpu')
model.load_state_dict(loaded['model_state_dict'])
opt.load_state_dict(loaded['optimizer_state_dict'])
start_epoch = loaded['epoch'] + 1
print(f"Resuming from epoch {start_epoch}, restored loss {loaded['loss']}")
print("Full checkpoint save/load round-trip succeeded.")


Resuming from epoch 6, restored loss 0.123
Full checkpoint save/load round-trip succeeded.


## 9. Build a Small 'Pretrained' Source CNN

We train a small CNN on synthetic image-like tensors (8x8 grayscale), then treat it as the pretrained feature extractor.


In [4]:
class TinyCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 8x8->4x4
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 4x4->2x2
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(16*2*2, num_classes))
    def forward(self, x):
        return self.classifier(self.features(x))

def synth_images(n, seed=0):
    g = np.random.default_rng(seed)
    X = g.standard_normal((n, 1, 8, 8)) * 0.1
    # synthetic 'object' classes: a positive blob in corner A or corner B
    y = np.zeros(n, dtype=np.int64)
    for i in range(n):
        X[i, 0, 1:4, 1:4] += 1.0 if i % 2 == 0 else 0.0
        X[i, 0, 4:7, 4:7] += 1.0 if i % 2 == 1 else 0.0
        y[i] = 0 if i % 2 == 0 else 1
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

src_X, src_y = synth_images(800)
print("Source dataset:", src_X.shape, src_y.shape)


Source dataset: torch.Size([800, 1, 8, 8]) torch.Size([800])


## 10. Train the Source Model (='pretrained')


In [5]:
src_model = TinyCNN(num_classes=2)
crit = nn.CrossEntropyLoss()
opt = optim.Adam(src_model.parameters(), lr=0.005)
for epoch in range(20):
    opt.zero_grad()
    loss = crit(src_model(src_X), src_y)
    loss.backward(); opt.step()
    if (epoch + 1) % 10 == 0:
        print(f"src epoch {epoch+1:2d}: loss={loss.item():.4f}")
with torch.no_grad():
    src_acc = (src_model(src_X).argmax(dim=1) == src_y).float().mean().item()
print(f"Source (pretrained) accuracy: {src_acc:.3f}")
torch.save(src_model.state_dict(), '_tmp_runs/src_pretrained.pt')


src epoch 10: loss=0.2298


src epoch 20: loss=0.0047


Source (pretrained) accuracy: 1.000


## 11. Feature Extraction Transfer Learning

Freeze the pretrained features, replace the classifier head, and train only the new head on a different (shifted) task.


In [6]:
def target_images(n, seed=1):
    g = np.random.default_rng(seed)
    X = g.standard_normal((n, 1, 8, 8)) * 0.1
    # shifted: class 0 = top-left blob, class 1 = centered blob (domain shift)
    y = np.zeros(n, dtype=np.int64)
    for i in range(n):
        X[i, 0, 1:4, 1:4] += 1.0 if i % 2 == 0 else 0.0
        X[i, 0, 3:5, 3:5] += 1.2 if i % 2 == 1 else 0.0
        y[i] = 0 if i % 2 == 0 else 1
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

tar_X, tar_y = target_images(400)
print("Target dataset:", tar_X.shape, tar_y.shape)

# Feature extraction: reuse pretrained features, new head
fe_model = TinyCNN(num_classes=2)
fe_model.load_state_dict(torch.load('_tmp_runs/src_pretrained.pt', map_location='cpu'))
# freeze features only (not the classifier)
for p in fe_model.features.parameters():
    p.requires_grad = False
print("Frozen feature params:", sum(1 for p in fe_model.features.parameters() if p.requires_grad is False))
print("Trainable params (classifier only):", sum(p.numel() for p in fe_model.parameters() if p.requires_grad))


Target dataset: torch.Size([400, 1, 8, 8]) torch.Size([400])
Frozen feature params: 4
Trainable params (classifier only): 130


## 12. Train the New Head (Feature Extraction)


In [7]:
opt = optim.Adam(filter(lambda p: p.requires_grad, fe_model.parameters()), lr=0.01)
for epoch in range(15):
    opt.zero_grad()
    loss = crit(fe_model(tar_X), tar_y)
    loss.backward(); opt.step()
with torch.no_grad():
    acc = (fe_model(tar_X).argmax(dim=1) == tar_y).float().mean().item()
# Verify frozen weights unchanged
orig = torch.load('_tmp_runs/src_pretrained.pt', map_location='cpu')
frozen_unchanged = all(
    torch.equal(fe_model.features.state_dict()[k.split('.', 1)[1]], v)
    for k, v in orig.items() if k.startswith('features.')
)
print(f"Feature-extraction target accuracy: {acc:.3f}")
print("Pretrained (frozen) weights unchanged after training:", frozen_unchanged)


Feature-extraction target accuracy: 1.000
Pretrained (frozen) weights unchanged after training: True


## 13. Fine-Tuning (Unfreeze)

Unfreeze all layers and continue training with a SMALL learning rate to avoid destroying pretrained features.


In [8]:
ft_model = TinyCNN(num_classes=2)
ft_model.load_state_dict(torch.load('_tmp_runs/src_pretrained.pt', map_location='cpu'))
# unfreeze all
for p in ft_model.parameters():
    p.requires_grad = True
# discriminative LR: smaller for features, larger for head
opt = optim.Adam([
    {'params': ft_model.features.parameters(), 'lr': 0.0001},
    {'params': ft_model.classifier.parameters(), 'lr': 0.001},
])
for epoch in range(15):
    opt.zero_grad()
    loss = crit(ft_model(tar_X), tar_y)
    loss.backward(); opt.step()
with torch.no_grad():
    acc = (ft_model(tar_X).argmax(dim=1) == tar_y).float().mean().item()
print(f"Fine-tuned target accuracy: {acc:.3f}")
print("Fine-tuning adapts the features themselves with a low LR.")


Fine-tuned target accuracy: 1.000
Fine-tuning adapts the features themselves with a low LR.


## 14. Decision Guidance

| Strategy | Use When | Data | Compute |
|---|---|---|---|
| Feature extraction | Small data, similar domain | Hundreds | Low |
| Fine-tune top layers | Medium data, similar domain | Thousands | Medium |
| Full fine-tune | Large data or domain shift | Tens of thousands | High |
| From scratch | Very different domain / enough data | Very large | Very high |

## 15. Common Mistakes / Debugging

- Forgetting to freeze → overwrites pretrained features.
- High LR for fine-tuning → destroys knowledge (use 10-100x lower).
- Not saving optimizer state → can't resume.
- Device mismatch on load → use `map_location='cpu'`.

## 16. When NOT to Use

- Pretrained models when the target domain is very different and data is plentiful — train from scratch.

## 17. Challenge

Compare fine-tuning vs training-from-scratch on the same target data and observe who wins.


In [9]:
# Challenge: from-scratch vs fine-tune on target
scratch = TinyCNN(num_classes=2)
opt = optim.Adam(scratch.parameters(), lr=0.001)
for _ in range(15):
    opt.zero_grad(); crit(scratch(tar_X), tar_y).backward(); opt.step()
with torch.no_grad():
    s_acc = (scratch(tar_X).argmax(dim=1) == tar_y).float().mean().item()
print(f"From-scratch target accuracy: {s_acc:.3f}")
print("Fine-tuning (with pretrained features) usually wins on small target data.")


From-scratch target accuracy: 1.000
Fine-tuning (with pretrained features) usually wins on small target data.


## 18. Closed-Book Recall

Without looking back:

1. Why save `state_dict()` instead of the whole model?
2. What happens if you don't freeze pretrained layers?
3. How choose between feature extraction and fine-tuning?
4. Why lower LR for pretrained layers?

## 19. Teach-Back Questions

Explain to another person:

- How feature extraction differs from fine-tuning.
- Why checkpoints include optimizer state.

## 20. Summary

You saved/loaded checkpoints, built a small pretrained CNN, and compared feature extraction vs fine-tuning vs from-scratch.

## 21. Further Experiment

- Resume training from a checkpoint and confirm loss continues from saved point.
- Vary how many layers you freeze.

## 22. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: torch, numpy
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
